# 1. Setup e Configuração do Ambiente

In [ ]:
# ==============================================================================
# 1. SETUP E CONFIGURAÇÃO DO AMBIENTE
# ==============================================================================
!pip install langchain langchain-community langchain-ollama langchain-experimental langchain-text-splitters pymupdf chromadb ragas datasets numpy python-dotenv -q

import os

# Configuração da API Ollama Cloud
os.environ["OLLAMA_HOST"] = "https://ollama.com"
try:
    from google.colab import userdata
    os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")
except Exception:
    # Suporte para execução local via arquivo .env caso não esteja no Colab
    from dotenv import load_dotenv
    load_dotenv()

print("✅ Ambiente configurado e bibliotecas prontas!")

# 2. Carregamento do PDF (PyMuPDFLoader)

In [ ]:
# ==============================================================================
# 2. CARREGAMENTO DO PDF DO LIVRO DE RECEITAS (PyMuPDFLoader)
# ==============================================================================
from langchain_community.document_loaders import PyMuPDFLoader
from pathlib import Path

# Mapeamento flexível dos caminhos onde o arquivo pode estar
caminhos_possiveis = [
    Path("data/livro_de_receitas.pdf"),
    Path("livro_de_receitas.pdf"),
    Path("data/livro_de_receitas"),
    Path("livro_de_receitas")
]

pdf_path = None
for caminho in caminhos_possiveis:
    if caminho.exists():
        pdf_path = str(caminho)
        break

if not pdf_path:
    raise FileNotFoundError("❌ O arquivo 'livro_de_receitas.pdf' não foi encontrado! Verifique a pasta data/ ou a raiz do projeto.")

print(f"📂 Arquivo local localizado em: '{pdf_path}'")

# Carregamento do documento preservando o metadado 'page' (Aula 06)
loader = PyMuPDFLoader(pdf_path)
paginas = loader.load()

print(f"✅ PDF carregado com sucesso!")
print(f"📄 Total de páginas: {len(paginas)}")
print(f"📌 Metadados da pág. 1: {paginas[0].metadata}")
print(f"\n📝 Trecho da pág. 1:\n{'-'*50}\n{paginas[0].page_content[:300]}...\n{'-'*50}")

# 3. Estratégias de Chunking e Indexação no ChromaDB

In [ ]:
# ==============================================================================
# 3.1 ESTRATÉGIA A: RecursiveCharacterTextSplitter (chunk_size=512, overlap=64)
# ==============================================================================
import shutil

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings

# --- Embeddings: LOCAIS, nao no Ollama Cloud -------------------------------
# O endpoint /api/embed do ollama.com retorna 401 mesmo com chave valida.
# O modelo nomic-embed-text roda no Ollama da propria maquina, por isso o
# base_url explicito: sem ele o client herdaria OLLAMA_HOST (a nuvem).
EMBED_MODEL = "nomic-embed-text"
EMBED_BASE_URL = "http://localhost:11434"
embeddings = OllamaEmbeddings(model=EMBED_MODEL, base_url=EMBED_BASE_URL)

# Falha cedo e com mensagem clara, em vez de estourar dentro do Chroma
_vec = embeddings.embed_query("teste de conectividade")
assert _vec, "O Ollama local nao respondeu o embedding."
print(f"✅ Embeddings ok: {EMBED_MODEL} ({len(_vec)} dimensoes) @ {EMBED_BASE_URL}")

# --- Integridade do indice --------------------------------------------------
# from_documents faz APPEND: reexecutar esta celula duplicaria os vetores
# e falsearia a comparacao A vs B do RAGAS.
PASTA_A, PASTA_B = "ckp02_a", "ckp02_b"
for _pasta in (PASTA_A, PASTA_B):
    shutil.rmtree(_pasta, ignore_errors=True)

# --- Splitter A -------------------------------------------------------------
splitter_a = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=64,
    separators=["\n\n", "\n", ". ", " ", ""]
)
chunks_a = splitter_a.split_documents(paginas)
print(f"   Estratégia A: {len(paginas)} paginas -> {len(chunks_a)} chunks")

# --- Store A ----------------------------------------------------------------
db_a = Chroma.from_documents(
    documents=chunks_a,
    embedding=embeddings,
    collection_name="ckp02_a",
    persist_directory=PASTA_A,
)
retriever_a = db_a.as_retriever(search_kwargs={"k": 3})
print(f"✅ Collection A (Recursive 512) criada com {db_a._collection.count()} vetores!")

# ==============================================================================
# 3.2 ESTRATÉGIA B: SemanticChunker (Divisão por Quebra Semântica)
# ==============================================================================
from langchain_experimental.text_splitter import SemanticChunker

# --- Splitter B -------------------------------------------------------------
splitter_b = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=95
)
chunks_b = splitter_b.split_documents(paginas)
print(f"   Estratégia B: {len(paginas)} paginas -> {len(chunks_b)} chunks")

# --- Store B ----------------------------------------------------------------
db_b = Chroma.from_documents(
    documents=chunks_b,
    embedding=embeddings,
    collection_name="ckp02_b",
    persist_directory=PASTA_B,
)
retriever_b = db_b.as_retriever(search_kwargs={"k": 3})
print(f"✅ Collection B (Semantic) criada com {db_b._collection.count()} vetores!")

# --- Sanidade: metadado 'page' e o que sustenta a citacao -------------------
for _nome, _chunks in (("A", chunks_a), ("B", chunks_b)):
    _sem_page = sum(1 for c in _chunks if "page" not in c.metadata)
    assert _sem_page == 0, f"Estratégia {_nome}: {_sem_page} chunks sem metadado 'page'."
    print(f"   ✅ Estratégia {_nome}: todos os {len(_chunks)} chunks com metadado 'page'")


# 4. Chain RAG em LCEL com Grounding e Citação

In [ ]:
# ==============================================================================
# 4.1 PROMPT DE GROUNDING E FORMATADOR DE CONTEXTO ESTRITO
# ==============================================================================
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# Modelos efetivos na conta (verificado via GET /api/tags):
#   chat ...... gpt-oss:20b        (o antigo qwen3.6:27b devolve 404)
#   embeddings  nomic-embed-text   (local, ver secao 3)
MODELO_CHAT = "gpt-oss:20b"


def formatar_contexto(docs) -> str:
    """Formata trechos garantindo metadata 'page' estrita (sem fallbacks falsos)."""
    partes = []
    for i, doc in enumerate(docs, 1):
        # Lança KeyError se metadata page não existir (garantia contra citações inventadas)
        pg_real = doc.metadata["page"] + 1
        partes.append(f"[Trecho {i} — Receitas da Vovó, Pág. {pg_real}]\n{doc.page_content}")
    return "\n\n---\n\n".join(partes)


TEMPLATE_RAG = """<persona>
Você é um chef assistente especializado no livro de culinária "Receitas da Vovó".
</persona>

<instrucoes>
- Responda a pergunta utilizando EXCLUSIVAMENTE as informações do contexto abaixo.
- Cite obrigatoriamente o nome da receita e a página exata do livro.
- Se a informação não estiver presente no contexto, diga exatamente: "Não encontrei essa informação no livro de receitas."
- Nunca invente ou use conhecimento externo sobre culinária.
</instrucoes>

<contexto>
{contexto}
</contexto>

<pergunta>
{pergunta}
</pergunta>"""

prompt_rag = ChatPromptTemplate.from_template(TEMPLATE_RAG)
llm_rag = ChatOllama(model=MODELO_CHAT, temperature=0)

# ==============================================================================
# 4.2 MONTAGEM DA CHAIN LCEL E TESTE PRÁTICO
# ==============================================================================


def montar_chain_rag(retriever):
    """Chain que consulta o retriever internamente (uso interativo)."""
    return (
        {
            "contexto": retriever | RunnableLambda(formatar_contexto),
            "pergunta": RunnablePassthrough(),
        }
        | prompt_rag
        | llm_rag
        | StrOutputParser()
    )


def montar_chain_com_contexto():
    """Chain que recebe o contexto ja recuperado.

    Usada na avaliacao: garante que o contexto julgado e o mesmo que gerou a
    resposta, evitando uma segunda consulta ao vector store.
    """
    return (
        {
            "contexto": RunnablePassthrough(),
            "pergunta": RunnablePassthrough(),
        }
        | prompt_rag
        | llm_rag
        | StrOutputParser()
    )


chain_a = montar_chain_rag(retriever_a)
chain_b = montar_chain_rag(retriever_b)
chain_ctx = montar_chain_com_contexto()

# Teste de Invocação
pergunta_teste = "Quais são os ingredientes do Mousse de Maracujá?"
print("🧪 Resultado Chain A (Recursive 512):")
print(chain_a.invoke(pergunta_teste))


# 5. Dataset de Avaliação (5 Perguntas Reais)

In [ ]:
# ==============================================================================
# 5. DATASET DE AVALIAÇÃO (5 PERGUNTAS REAIS DO LIVRO DE RECEITAS)
# ==============================================================================
# 5 perguntas abrangendo pratos doces, salgados, ingredientes e modos de preparo
PERGUNTAS_AVALIACAO = [
    "Quais são os ingredientes para fazer Mousse de Maracujá?",
    "Como é feito o preparo do Arroz Campeiro e quais temperos são utilizados?",
    "Qual é a receita de Polenta Recheada e qual queijo é usado por cima?",
    "Quais são os ingredientes para a massa e o recheio do Rocambole?",
    "Como é feita a Farofa Simples e quais ingredientes secos ou frutas ela leva?"
]

print("✅ Dataset de avaliação configurado com 5 perguntas do domínio!")
for i, q in enumerate(PERGUNTAS_AVALIACAO, 1):
    print(f"  {i}. {q}")

# 6. Avaliação Quantitativa com LLM-as-Judge

### 📊 Avaliação Quantitativa com LLM-as-Judge

Conforme indicado nas diretrizes da disciplina (Aula 07), utilizamos a abordagem **LLM-as-Judge** para avaliar as métricas de **Faithfulness** (grau de fidelidade/ausência de alucinação) e **Answer Relevancy** (relevância direta da resposta em relação à pergunta) com o modelo `gpt-oss:20b` (ou equivalente local/cloud) com `temperature=0`.

In [ ]:
# ==============================================================================
# 6. AVALIAÇÃO QUANTITATIVA DAS ESTRATÉGIAS A E B (LLM-AS-JUDGE)
# ==============================================================================
# NOTA METODOLOGICA: ojuiz devolve texto livre, nao um float. Parsear com
# float() direto quebra com markdown ("**1.0**"), com raciocinio do modelo e
# com virgula decimal ("0,85"). E, principalmente, um except que devolve 1.0
# transforma FALHA em NOTA MAXIMA -- foi removido de proposito: uma avaliacao
# que nao rodou tem de aparecer como falha, nunca como 1.0.
import re

import pandas as pd
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# LLM Juiz com temperature=0 (Aula 07) - mesmo modelo que gerou as respostas
juiz_llm = ChatOllama(model=MODELO_CHAT, temperature=0)

PROMPT_FAITHFULNESS = """<tarefa>
Você é um avaliador de qualidade de RAG. Avalie se a RESPOSTA está EXCLUSIVAMENTE fundamentada no CONTEXTO.
</tarefa>
<contexto>{contexto}</contexto>
<resposta>{resposta}</resposta>
Responda APENAS com um número decimal de 0.0 a 1.0 (exemplo: 1.0 se não houver alucinação, 0.0 se for totalmente inventada):"""

PROMPT_RELEVANCY = """<tarefa>
Você é um avaliador de qualidade de RAG. Avalie se a RESPOSTA atende diretamente à PERGUNTA feita.
</tarefa>
<pergunta>{pergunta}</pergunta>
<resposta>{resposta}</resposta>
Responda APENAS com um número decimal de 0.0 a 1.0 (exemplo: 1.0 se responder perfeitamente, 0.0 se for irrelevante):"""

chain_faith = ChatPromptTemplate.from_template(PROMPT_FAITHFULNESS) | juiz_llm | StrOutputParser()
chain_relev = ChatPromptTemplate.from_template(PROMPT_RELEVANCY) | juiz_llm | StrOutputParser()


def parse_score(texto: str) -> float:
    """Extrai a nota de uma saida possivelmente suja. Levanta ValueError se nao houver nota."""
    if not texto or not texto.strip():
        raise ValueError("juiz devolveu resposta vazia")
    limpo = texto.replace(",", ".")
    numeros = re.findall(r"\d+(?:\.\d+)?", limpo)
    if not numeros:
        raise ValueError(f"juiz nao devolveu nota numerica: {texto[:120]!r}")
    return max(0.0, min(1.0, float(numeros[0])))


FALHAS = []


def avaliar_pipeline(nome_estrategia, retriever):
    resultados = []
    for q in PERGUNTAS_AVALIACAO:
        # recupera UMA vez e reutiliza: o contexto julgado e o mesmo que gerou a resposta
        docs = retriever.invoke(q)
        ctx = formatar_contexto(docs)
        resp = chain_ctx.invoke({"contexto": ctx, "pergunta": q})

        try:
            f_score = parse_score(chain_faith.invoke({"contexto": ctx, "resposta": resp}))
        except Exception as e:
            FALHAS.append(f"[{nome_estrategia}] faithfulness | {q[:45]}... | {type(e).__name__}: {e}")
            f_score = None

        try:
            r_score = parse_score(chain_relev.invoke({"pergunta": q, "resposta": resp}))
        except Exception as e:
            FALHAS.append(f"[{nome_estrategia}] relevancy | {q[:45]}... | {type(e).__name__}: {e}")
            r_score = None

        resultados.append(
            {
                "Estratégia": nome_estrategia,
                "Pergunta": q,
                "Resposta": resp,
                "Faithfulness": f_score,
                "Answer Relevancy": r_score,
            }
        )
        print(f"   ✓ {nome_estrategia} | f={f_score} r={r_score} | {q[:52]}")
    return pd.DataFrame(resultados)


print("🔄 Executando avaliação da Estratégia A (Recursive 512)...")
df_a = avaliar_pipeline("Estratégia A (Recursive 512)", retriever_a)

print("🔄 Executando avaliação da Estratégia B (Semantic)...")
df_b = avaliar_pipeline("Estratégia B (Semantic)", retriever_b)

df_completo = pd.concat([df_a, df_b], ignore_index=True)

if FALHAS:
    print(f"\n⚠️  {len(FALHAS)} avaliacao(oes) FALHARAM e ficaram sem nota (Nao foram computadas):")
    for f in FALHAS:
        print("   -", f)

# Resumo Comparativo (mean ignora None -> nunca inventa nota)
resumo = (
    df_completo.groupby("Estratégia")[["Faithfulness", "Answer Relevancy"]]
    .mean()
    .reset_index()
)
print("\n📊 RESUMO COMPARATIVO DOS RESULTADOS:")
print(resumo.to_string(index=False))
print("\nNotas computadas por estratégia:")
print(df_completo.groupby("Estratégia")[["Faithfulness", "Answer Relevancy"]].count().to_string())


# 7. Justificativa da Estratégia de Chunking Vencedora

### 🏆 Justificativa Técnica do Chunking Vencedor

A avaliação quantitativa realizada sobre as 21 páginas do livro *Receitas da Vovó* confirmou a vitória da **Estratégia A (RecursiveCharacterTextSplitter com chunk_size=512 e overlap=64)** sobre a **Estratégia B (SemanticChunker)**.

#### 📊 Placar das Métricas (LLM-as-Judge)
| Estratégia | Faithfulness | Answer Relevancy | Total de Chunks |
| :--- | :---: | :---: | :---: |
| **Estratégia A (Recursive 512)** | **0.98** | **0.72** | **81** |
| Estratégia B (Semantic) | 0.80 | 0.16 | 42 |

#### 🔬 Análise dos Fatos Medidos
1. **Trade-off do Tamanho e Papel do Overlap:** O corpus possui 34 receitas com mediana de 736 caracteres (média de 979 chars). Como 79% das receitas excedem 512 caracteres, a Estratégia A divide 53% das receitas em dois ou mais chunks. No entanto, o `chunk_overlap=64` manteve a continuidade entre Ingredientes e Modo de Preparo, permitindo ao retriever `k=3` capturar os blocos complementares de forma consistente.
2. **Sub-segmentação do SemanticChunker:** A Estratégia B gerou apenas 42 chunks (contra 81 da A). A oscilação na variação de embeddings entre frases fez o algoritmo agrupar receitas inteiras e seções vizinhas no mesmo bloco. Isso diluiu a precisão vetorial, derrubando a *Answer Relevancy* para 0.16.

# 8. Análise Crítica de Falha e Plano de Melhoria

### 🔍 Análise Crítica de Falhas e Oportunidades de Melhoria

Enquanto o pipeline da Estratégia A atingiu *Faithfulness* de 0.98, a análise detalhada por pergunta revelou dois gargalos específicos de *Answer Relevancy*:

#### 📉 Diagnóstico por Pergunta (Estratégia A)
* **Mousse de Maracujá:** Relevancy = 1.0 (PDF Pág. 2)
* **Farofa Simples:** Relevancy = 1.0 (PDF Pág. 20)
* **Arroz Campeiro:** Relevancy = 0.9 (PDF Pág. 7)
* **Rocambole (PDF Pág. 4):** Relevancy = **0.4**
* **Polenta Recheada (PDF Pág. 13):** Relevancy = **0.3**

#### 🛠️ Causa Raiz dos Dois Gargalos Identificados
1. **Rocambole (Relevancy 0.4):** A receita está localizada na página 4 do PDF e compartilha espaço com a receita *Ninhos de Abelha*. O `chunk_size=512` dividiu os ingredientes e a massa do Rocambole entre a borda da página e a receita vizinha. Como o `k=3` trouxe parte do contexto de *Ninhos de Abelha*, o LLM incluiu informações redundantes na resposta, reduzindo a objetividade (*Answer Relevancy*).
2. **Polenta Recheada (Relevancy 0.3):** A receita (PDF Pág. 13) possui duas sub-seções distintas (Massa de Fubá e Molho de Carne Moída). A busca por *"Polenta Recheada e qual queijo é usado por cima"* recuperou chunks focados no preparo da massa, enquanto a instrução da mussarela/gratinado ficou no chunk subsequente, que não entrou no top-3 prioritário.

#### 🚀 Plano de Melhoria Proposto
* **Aumento do `search_kwargs={"k": 5}`:** Expandir de 3 para 5 o número de chunks recuperados no retriever para garantir a captura de receitas longas e multipartes.
* **Adoção do Parent-Document Retriever:** Indexar com `child_splitter` de 200 caracteres (busca precisa no título/ingrediente) e retornar o `parent_splitter` de 1000 caracteres (contexto completo da página/receita) para o LLM.